# FDN Frequency Response: FLAMO Probe vs FLAMO Core Forward

This example compares two ways to obtain the frequency response **from the same FLAMO model**:

1. **FLAMO probe on unit circle**: Call `model.probe(z)` at each z = exp(jω) on the unit circle. This evaluates H(z) directly.
2. **FLAMO core forward**: Extract the core (without Shell), create an array of ones with shape `(1, nfft//2+1, n_in)` (unit amplitude at each frequency bin), and pass it through `core.forward()`. The output is the frequency response H(ω) directly — no FFT needed.

Both should give the same result.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

import pyFDN

np.random.seed(42)
Fs = 48000
nfft = 2**15

N = 4
delays = np.array([101, 157, 211, 263], dtype=int)
g = 0.999
A = pyFDN.random_orthogonal(N) @ np.diag(g**delays)
b = np.random.randn(N, 1)
c = np.random.randn(1, N)
d = np.array([[0.1]])

model = pyFDN.dss_to_flamo(A, b, c, d, delays, Fs, nfft=nfft, dtype=torch.float64)

print(f"N={N}, delays={delays.tolist()}, nfft={nfft}")

## Method 1: FLAMO probe on unit circle

Call `model.probe(z)` at each z = exp(2πjk/nfft) to get H(z) directly.

In [ ]:
n_freq = nfft // 2 + 1
h_probe = np.zeros(n_freq, dtype=np.complex128)

for k in range(n_freq):
    z = np.exp(2j * np.pi * k / nfft)
    zt = torch.tensor(z, dtype=torch.complex128)
    with torch.no_grad():
        H_k = model.probe(zt)
    h_probe[k] = H_k.squeeze().cpu().numpy()

w = np.linspace(0, np.pi, n_freq)
print(f"H_probe shape: {h_probe.shape}")

## Method 2: FLAMO core forward

Extract the core (bypassing the Shell), then pass an array of ones `(1, nfft//2+1, 1)` through the core. The core operates in frequency domain, so ones = unit amplitude at each bin; the output is H(ω) directly.

In [ ]:
core = model.get_core()
x_ones = torch.ones(1, n_freq, 1, dtype=torch.complex128)
with torch.no_grad():
    h_core_forward = core(x_ones)
h_flamo_forward = h_core_forward.squeeze().cpu().numpy()

print(f"h_flamo_forward shape: {h_flamo_forward.shape}")

## Compare results

In [ ]:
f_hz = w / np.pi * (Fs / 2)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(f_hz, 20 * np.log10(np.abs(h_probe) + 1e-12), label="FLAMO probe (unit circle)")
axes[0].plot(f_hz, 20 * np.log10(np.abs(h_flamo_forward) + 1e-12), ls="--", alpha=0.8, label="FLAMO core forward")
axes[0].set_ylabel("Magnitude [dB]")
axes[0].set_title("Frequency response")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

diff_mag = np.abs(h_probe - h_flamo_forward)
axes[1].semilogy(f_hz, diff_mag)
axes[1].set_ylabel("|H_probe - H_forward|")
axes[1].set_xlabel("Frequency [Hz]")
axes[1].set_title("Difference between methods")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Assert match

In [ ]:
max_diff = np.max(np.abs(h_probe - h_flamo_forward))
rel_diff = np.max(np.abs(h_probe - h_flamo_forward) / (np.abs(h_probe) + 1e-12))

print(f"Max absolute difference: {max_diff:.2e}")
print(f"Max relative difference: {rel_diff:.2e}")

assert max_diff < 5e-3, "FLAMO probe and FLAMO core forward should match"
print("\n✓ Both methods give the same result.")